# 电商 3-arm A/B 测试分析 · 3-arm E-commerce A/B Test

> **实验设计 · Design**: 三臂对照 (Control / Variant_A / Variant_B)，单元为 session。
> **零假设 H0**: 某 Variant 与 Control 的 session 转化率相同。
> **检验 Tests**: 双尾两比例 z 检验 × 2，Bonferroni 校正后 α = 0.025。

## 1. 环境 · Environment

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import (
    proportions_ztest, proportion_confint,
    confint_proportions_2indep, proportion_effectsize,
)
from statsmodels.stats.power import NormalIndPower

plt.rcParams.update({'figure.figsize': (8, 4.5), 'figure.dpi': 110})
pd.set_option('display.max_columns', 20)

## 2. 加载数据 · Load data
`events.csv` 包含 200 万条用户行为事件，覆盖 2021-01-01 ~ 2023-12-31。

In [ ]:
ev = pd.read_csv('../1/events.csv', parse_dates=['timestamp'])
ev['date'] = ev['timestamp'].dt.date
print('events   :', len(ev))
print('sessions :', ev['session_id'].nunique())
print('customers:', ev['customer_id'].nunique())
print('window   :', ev['timestamp'].min(), '->', ev['timestamp'].max())
ev.head()

## 3. 数据质量 · Data quality

两个主要发现：

1. **分组污染 Assignment contamination**: 约 **65%** 的 session 包含多个 `experiment_group` 标签，
   违反了 A/B 测试的单元内一致性假设。本项目使用 **modal（众数法）** 分组：
   取 session 内出现最多的 group。
2. **traffic_source 大小写不一致**: `Organic` 与 `ORGANIC`、`Email` 与 `EMAIL` 等同时存在。

In [ ]:
# Contamination check
gc = ev.groupby(['session_id', 'experiment_group']).size().unstack(fill_value=0)
contaminated = (gc > 0).sum(axis=1) > 1
print('Total sessions :', len(gc))
print('Contaminated   :', contaminated.sum(), '(', f'{contaminated.mean():.1%}', ')')

# Traffic source DQ
print('\nTraffic-source case duplicates:')
print(ev['traffic_source'].value_counts())

## 4. 分组与转化定义 · Assignment & outcome

In [ ]:
modal_group = gc.idxmax(axis=1)
purchased = ev.groupby('session_id')['event_type'].apply(lambda s: (s == 'purchase').any())

session = pd.DataFrame({
    'group': modal_group,
    'converted': purchased.astype(int).reindex(modal_group.index, fill_value=0),
}).reset_index()

summary = session.groupby('group')['converted'].agg(['count', 'sum', 'mean'])
summary.columns = ['n', 'conversions', 'rate']
summary

## 5. 假设检验 · Hypothesis tests

对每个 variant 与 Control 做双尾两比例 z 检验，共 2 次比较，使用 Bonferroni 校正 (α_family = 0.05 → α_adj = 0.025)。

In [ ]:
ALPHA = 0.05
N_COMP = 2
ALPHA_ADJ = ALPHA / N_COMP

def test(variant):
    g = session.groupby('group')['converted'].agg(['count', 'sum', 'mean'])
    x_c = int(g.loc['Control', 'sum'])
    n_c = int(g.loc['Control', 'count'])
    p_c = g.loc['Control', 'mean']
    x_v = int(g.loc[variant, 'sum'])
    n_v = int(g.loc[variant, 'count'])
    p_v = g.loc[variant, 'mean']
    z, p = proportions_ztest([x_v, x_c], [n_v, n_c], alternative='two-sided')
    lo, hi = confint_proportions_2indep(x_v, n_v, x_c, n_c, method='wald')
    lift = (p_v - p_c) / p_c * 100
    print(f'{variant} vs Control: z={z:.3f}  p={p:.2e}  '
          f'p_bonf={min(p*N_COMP, 1):.2e}  '
          f'lift={lift:+.2f}%  '
          f'95% CI(diff)=[{lo*100:+.2f}, {hi*100:+.2f}] pp')

test('Variant_A')
test('Variant_B')

**结论 · Interpretation**

- **Variant_A** 相比 Control 转化率下降 **-1.99 pp** (相对 -12.98%)，p = 2.92e-45，Bonferroni 校正后仍高度显著，**方向为负**。
- **Variant_B** 相比 Control 转化率下降 **-0.54 pp** (相对 -3.53%)，p = 5.88e-04，同样显著，**方向也为负**。
- 两个 Variant 的差异 95% CI 都完全位于零左侧，没有任何新版优于旧版的证据。

## 6. 效应量与功效 · Effect size and power

In [ ]:
for v in ['Variant_A', 'Variant_B']:
    p_c = session[session['group'] == 'Control']['converted'].mean()
    p_v = session[session['group'] == v]['converted'].mean()
    n_v = (session['group'] == v).sum()
    n_c = (session['group'] == 'Control').sum()
    h = proportion_effectsize(p_v, p_c)
    power = NormalIndPower().solve_power(
        effect_size=abs(h), nobs1=n_v, alpha=ALPHA_ADJ,
        ratio=n_c / n_v, alternative='two-sided')
    print(f'{v}: Cohen h = {h:+.4f}, observed power = {power:.2%}')

## 7. 业务结论 · Business conclusion

| 结论 · Finding | 依据 · Evidence |
|----------------|-----------------|
| **统计层面 Statistical** | 两个 Variant 的 Bonferroni 校正后 p 值均远小于 0.025，方向均为负 |
| **业务层面 Business** | Variant_A 相对下降 -12.98%、Variant_B 相对下降 -3.53% |
| **功效层面 Power** | Variant_A 功效 = 100%，Variant_B = 89%，样本量充足 |
| **最终建议 · Recommendation** | **保留 Control，两个 Variant 均不建议上线 · Keep Control; do NOT launch either variant** |

**附注 · Data-quality caveats**: 65% 的 session 存在分组污染，应先修复埋点/分流逻辑（保证同一 session 只打一次分组标签）。

**附注**: `traffic_source` 字段大小写不一致（如 `Organic` 与 `ORGANIC`），建议在数据摄入层统一小写。